<a href="https://colab.research.google.com/github/JPG27784/Machine-Learning/blob/main/Multi_Agent_Systems_bp_03152026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agent Systems

#### THIS SECTION OF CODE IS COVERED IN THE EARLIER CLASS. BRIEF REVISION & SET UP THE GROUND FOR THE MULTI-AGENTIC AI POWERED LLMs.
-----------------------------------------------------------------------------

*A basic chatbot that can answer customer queries*

## Problem Statement
Shopping online can be overwhelming. You search for a simple pair of shoes, but end up scrolling through countless options—many irrelevant, some too expensive, others just not right. Traditional search engines rely on keywords, often missing what you truly need.

Let's build an AI-powered product discovery chatbot changes this. Using advanced language models and vector-based search, it goes beyond keywords to understand your intent, offering personalized, context-aware recommendations in seconds.

<center><img src="https://www.pranathiss.com/static/assets/images/ai-powered-chatBot.webp" width=500/></center>

This smart solution enhances the shopping experience, increasing customer satisfaction, engagement, and conversions. The future of e-commerce is here—smarter, intuitive, and built for you.

### What we did already??
- A conversational chatbot that answers basic queries & is made contextually aware with the help of RAG & Vector Stores.


### What are we going to do today?
- Add agents to the chatbot to enhance its features.


In [ ]:
# Installing the packages used in this notebook
# Tip: after this cell finishes in Colab, restart the runtime once before running the rest.
!pip install -q -U     langchain   langsmith  langchain-classic     langchain-openai     langchain-community     langchain-text-splitters     langchainhub     faiss-cpu     tavily-python     beautifulsoup4     gradio     python-dotenv     kagglehub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.3/378.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.6/19.6 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take 

In [ ]:
# Optional: print versions so students can see the environment they are using
import importlib
for pkg in [
    "langchain", "langchain_classic", "langchain_openai",
    "langchain_community", "langchain_text_splitters"
]:
    try:
        module = importlib.import_module(pkg)
        print(f"{pkg}: {getattr(module, '__version__', 'version not exposed')}")
    except Exception as e:
        print(f"{pkg}: not available ({e})")


langchain: 1.2.15
langchain_classic: 1.0.4
langchain_openai: version not exposed
langchain_community: 0.4.1
langchain_text_splitters: version not exposed


In [ ]:
!pip install -U langsmith

In [ ]:
# Necessary imports
import csv
import getpass
import math
import os

import kagglehub
import numpy as np
import pandas as pd
from google.colab import drive

from langchain_classic import hub
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.vectorstores import FAISS
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from getpass import getpass


In [ ]:
# Setting up Open AI key
os.environ["OPENAI_API_KEY"] = getpass()

··········


In [ ]:
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
print(f"OPENAI_API_KEY: {OPENAI_API_KEY[0:2]}...")

OPENAI_API_KEY: sk...


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
project_dir='/content/drive/MyDrive/IK/' #Replace with your path
env_file_path=os.path.join('/content/drive/MyDrive/IK/BuildingApplicationsWithLLMsAgents-Lite/DemoNotebooks/.env')
print(f"The .env file is located at: {env_file_path}")

The .env file is located at: /content/drive/MyDrive/IK/BuildingApplicationsWithLLMsAgents-Lite/DemoNotebooks/.env


Loads secrets/config (e.g., API keys) into the environment so downstream clients initialize correctly.

**Key functions/classes (what they do)**
- `load_dotenv(path)` loads key/value pairs from a `.env` file into the process environment.
- `os.getenv('VAR')` reads an environment variable; returns `None` if missing.
- Make sure required keys (e.g., `OPENAI_API_KEY`) are set *before* initializing any LLM/LangChain clients.
- `os.environ['VAR'] = ...` sets an environment variable for the current process (and children).

**Outputs / artifacts**
- Defines/updates: `OPENAI_API_KEY`

In [ ]:
# Loading the data
# IMPORTANT: make sure sample_dataset.csv exists inside project_dir.
dataset_path = os.path.join(project_dir, "sample_dataset.csv")
print("Looking for dataset at:", dataset_path)

if not os.path.exists(dataset_path):
    raise FileNotFoundError(
        f"Could not find the dataset at {dataset_path}. "
        "Upload/copy sample_dataset.csv into project_dir first."
    )

df = pd.read_csv(dataset_path, index_col=0)
print("Loaded rows:", len(df))

# Preparing product descriptions for retrieval
product_description = []
product_description_len = []

# Keep the first 100 rows for a lightweight classroom demo
for _, row in df.head(100).iterrows():
    title = row.get("TITLE", None)
    description = row.get("DESCRIPTION", None)

    parts = []
    if pd.notna(title):
        parts.append(f"Title {title}")
    if pd.notna(description):
        parts.append(f"Description {description}")

    product = "\n\n".join(parts).strip()
    if product:
        product_description.append(product)
        product_description_len.append(len(product))

print("Number of product documents:", len(product_description))

Looking for dataset at: /content/drive/MyDrive/IK/sample_dataset.csv
Loaded rows: 100
Number of product documents: 100


In [ ]:
df

,TITLE,BULLET_POINTS,DESCRIPTION,PRODUCT_TYPE_ID,PRODUCT_LENGTH
PRODUCT_ID,,,,,
1925202,ArtzFolio Tulip Flowers Blackout Curtain for D...,[LUXURIOUS & APPEALING: Beautiful custom-made ...,NaN,1650,2125.980000
2673191,Marks & Spencer Girls' Pyjama Sets T86_2561C_N...,"[Harry Potter Hedwig Pyjamas (6-16 Yrs),100% c...",NaN,2755,393.700000
2765088,PRIKNIK Horn Red Electric Air Horn Compressor ...,"[Loud Dual Tone Trumpet Horn, Compatible With ...","Specifications: Color: Red, Material: Aluminiu...",7537,748.031495
1594019,ALISHAH Women's Cotton Ankle Length Leggings C...,[Made By 95%cotton and 5% Lycra which gives yo...,AISHAH Women's Lycra Cotton Ankel Leggings. Br...,2996,787.401574
283658,The United Empire Loyalists: A Chronicle of th...,NaN,NaN,6112,598.424000
...,...,...,...,...,...
107333,Carl Pops Up,NaN,NaN,40,800.000000
2919319,Generic Chiffon printed dupatta with Golden do...,"[Fabric: chiffon, size: 2.25 meters, fancy dup...","Fancy dupatta border golden Colour, light in w...",2918,8858.250000
90582,CAUGHT IN THE ACT (Loveswept),NaN,NaN,3383,450.000000


In [ ]:

# Chunk the text so retrieval works better
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)
documents = text_splitter.create_documents(product_description)
print("Number of chunks:", len(documents))

# Build embeddings + vector store
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector = FAISS.from_documents(documents, embeddings)
retriever = vector.as_retriever(search_kwargs={"k": 4})

# Base chat model used across the notebook
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Build a simple RAG chain for the Amazon dataset
rag_prompt = ChatPromptTemplate.from_template("""
Answer the following question using only the provided context.
If the answer is not in the context, say you could not find it in the Amazon dataset.

<context>
{context}
</context>

Question: {input}
""")

document_chain = create_stuff_documents_chain(llm, rag_prompt)
retrieval_chain = create_retrieval_chain(retriever, document_chain)

Number of chunks: 240


-----------------------------------------------------------------------------
## TODAY'S CONTENT BEGINS HERE
-----------------------------------------------------------------------------

### Limitations of the LLMs

Large Language Models (LLMs) like ChatGPT, GPT-4, and other AI systems are incredibly powerful, but they have a key limitation: they rely on pre-trained knowledge, which becomes outdated over time. This is where Tavily plays a crucial role by integrating real-time web search capabilities to keep AI responses up-to-date and contextually relevant.

#### How does Tavily improves it?
- Tavily enables live searches to retrieve the latest news, research papers, and real-world updates.
- LLMs have a knowledge cutoff (e.g., ChatGPT may not know recent events). Tavily enables AI to bridge this gap by fetching real-time data from the web.

So, let's set up the API key for using this in our project.
It's FREE!!!!!!

In [ ]:
# Setting up the Tavily API Key to do web search
os.environ["TAVILY_API_KEY"] = getpass()

··········


In [ ]:
# Define a tool for Amazon product search
@tool
def amazon_product_search(query: str) -> str:
    """Search the local Amazon product dataset.
    Use this tool for questions about products, descriptions, and Amazon catalog items in the demo data."""

    docs = retriever.invoke(query)
    if not docs:
        return "No relevant Amazon product information was found in the local dataset."

    formatted_docs = []
    for i, doc in enumerate(docs, start=1):
        formatted_docs.append(f"Result {i}: {doc.page_content}")
    return "\n\n".join(formatted_docs)


@tool
def add(a: str, b: str) -> str:
    """Add two numbers. Make sure to use this for addition"""
    return str(int(a) + int(b))

In [ ]:
print("Name:", amazon_product_search.name)
print("Description", amazon_product_search.description)
print("args", amazon_product_search.args)

Name: amazon_product_search
Description Search the local Amazon product dataset.
    Use this tool for questions about products, descriptions, and Amazon catalog items in the demo data.
args {'query': {'title': 'Query', 'type': 'string'}}


In [ ]:
@tool
def google_search(query: str) -> str:
    """Search Google for recent results."""
    pass

### Code Explanation:

- We turn a normal Python function into a **tool** using `@tool`.
- This tool calls the **retriever** built from the FAISS vector store.
- The agent can now decide when to query the local Amazon dataset during its ReAct loop.
- Returning a clean text block makes the tool output easier for students to read in the agent trace.


In [ ]:
# Tavily web-search tool
@tool
def search_tavily(query: str) -> str:
    """Search the web for fresh information when the local Amazon dataset is not enough."""

    search_tool = TavilySearchResults(
        max_results=5,
        include_answer=True,
        include_raw_content=False,
        include_images=False,
        search_depth="advanced",
    )

    results = search_tool.invoke(query)
    if not results:
        return "No Tavily results were returned."

    cleaned = []
    for i, item in enumerate(results, start=1):
        url = item.get("url", "")
        content = item.get("content", "")
        cleaned.append(f"Web Result {i}: \n URL: {url} Content: {content}")

    return "\n\n".join(cleaned)


### Code Explanation:

- `search_tavily` gives the agent access to **fresh web information**.
- This is useful when the local Amazon dataset is incomplete, outdated, or too noisy.
- In class, this is a nice example of how ReAct agents can choose between **multiple external tools**.


In [ ]:
# Create the list of tools the ReAct agent can use
tools = [search_tavily, amazon_product_search]

In [ ]:
# Imports for the ReAct agent section
# We keep this section explicit because it is useful for teaching.
print("Agent-related imports were loaded in the main import cell.")
print("Using:")
print("- hub for the prompt")
print("- create_react_agent for the ReAct loop")
print("- AgentExecutor for execution + verbose traces")
print("- RunnableWithMessageHistory for conversation memory")


Agent-related imports were loaded in the main import cell.
Using:
- hub for the prompt
- create_react_agent for the ReAct loop
- AgentExecutor for execution + verbose traces
- RunnableWithMessageHistory for conversation memory


In [ ]:
# Creating a Prompt Template + Memory
# We first try to pull the classic ReAct chat prompt from LangChain Hub.
# If Hub is unavailable, we fall back to an equivalent local prompt.

try:
    prompt = hub.pull("hwchase17/react-chat")
    print("Loaded prompt from LangChain Hub: hwchase17/react-chat")
except Exception as e:
    print(f"Hub prompt could not be loaded ({e}). Using a local fallback prompt instead.")
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant that follows the ReAct pattern. Use tools when needed. "
                   "When you know the answer, provide a final answer clearly."),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),  ("ai", "Thought: I should think step by step and use a tool if needed. {agent_scratchpad}"),
    ])

# In-memory conversation history for a single demo session
memory = InMemoryChatMessageHistory()



Loaded prompt from LangChain Hub: hwchase17/react-chat


### Code Explanation:
- `hub.pull("hwchase17/react-chat")` loads a classic **ReAct chat prompt**.
- That prompt helps the model follow the pattern: **reason → choose a tool → observe → continue → answer**.
- `InMemoryChatMessageHistory()` stores previous messages in RAM so follow-up questions can use context.
- This is a simple memory choice that works well for classroom demos.


In [ ]:
# Print the prompt so students can inspect it
if hasattr(prompt, "template"):
    print(prompt.template)
elif hasattr(prompt, "pretty_repr"):
    print(prompt.pretty_repr())
else:
    print(prompt)


Assistant is a large language model trained by OpenAI.

Assistant is designed to be able to assist with a wide range of tasks, from answering simple questions to providing in-depth explanations and discussions on a wide range of topics. As a language model, Assistant is able to generate human-like text based on the input it receives, allowing it to engage in natural-sounding conversations and provide responses that are coherent and relevant to the topic at hand.

Assistant is constantly learning and improving, and its capabilities are constantly evolving. It is able to process and understand large amounts of text, and can use this knowledge to provide accurate and informative responses to a wide range of questions. Additionally, Assistant is able to generate its own text based on the input it receives, allowing it to engage in discussions and provide explanations and descriptions on a wide range of topics.

Overall, Assistant is a powerful tool that can help with a wide range of tasks 

In [ ]:
# Create the ReAct agent
# We use ChatOpenAI instead of the legacy completion model because it is the most reliable option today.
# The ReAct idea stays the same: the model reasons, selects tools, observes results, and then answers.

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
)


### Code Explanation

1. **LLM**
   - We use `ChatOpenAI` as the reasoning model.
   - `temperature=0` makes outputs more deterministic and classroom-friendly.

2. **ReAct agent**
   - `create_react_agent(...)` builds the logic that decides when to use tools.
   - The agent follows the ReAct flow: **Thought → Action → Observation → Final Answer**.

3. **AgentExecutor**
   - Runs the agent.
   - `verbose=True` is important for teaching because it shows the trace of tool use.


### Why are we using `ChatOpenAI` here?

The original notebook used the legacy `OpenAI(...)` completion model for ReAct. In practice, that is harder to keep working reliably now because modern LangChain and modern OpenAI usage are centered on **chat models**. The teaching idea is still the same:

- the agent **reasons**,
- chooses a **tool**,
- reads the **observation**,
- and then gives a **final answer**.

So we keep the ReAct structure, the Hub prompt, the memory wrapper, and the verbose trace — but use a more reliable chat model underneath.


In [ ]:
# Wrap the agent with chat history so follow-up questions can use memory
agent_with_chat_history = RunnableWithMessageHistory(
    agent_executor,
    lambda session_id: memory,
    input_messages_key="input",
    history_messages_key="chat_history",
)


### Code Explanation:

- Initializing an Agent with chat history:
 - `session_id` is useful in real-world applications where session tracking is required. Here, it simply returns the in-memory chat history for simplicity.
 - `RunnableWithMessageHistory` ensures the agent retains context across multiple interactions. This is useful for multi-turn conversations where past messages influence future responses.


In [ ]:
# Invoke the agent with chat history
# Let's try a query based on recent data and check whether 'Tavily Search' is triggered
result = agent_with_chat_history.invoke(
    {"input": "What is the latest fashion trends in 2026?"},
    config={"configurable": {"session_id": "<foo>"}},
)



> Entering new AgentExecutor chain...
```
Thought: Do I need to use a tool? Yes
Action: search_tavily
Action Input: latest fashion trends 2026Web Result 1: 
 URL: https://www.wmagazine.com/fashion/fall-2026-fashion-trends Content: ## The Women of Wall Street

Following multiple seasons of oversize menswear silhouettes, this season brought a slimmer fit—sharply tailored dark suits appeared at Saint Laurent, Gucci, and Tom Ford. A fresh take for fall could also be found in the styling elements, like hats, coats worn over the shoulders, and a wealth of pinstripes.

From left to right: looks from Tom Ford, Gucci and Saint Laurent

From left to right: looks from Tom Ford, Gucci and Saint Laurent

## What’s Your Fetish?

Fall 2026 saw elements of “innocence” turned on their heads. Fetishwear crept into details like McQueen’s babydoll top, which was made of body armor. Meanwhile, one of Phoebe Philo’s dresses seemed to be uncovered in all the right places.

From left to right: looks from Mi

In [ ]:
print(result)
print(result.keys())

{'input': 'What is the latest fashion trends in 2026?', 'chat_history': [], 'output': 'The latest fashion trends for 2026 showcase a mix of styles and influences. Key trends include:\n\n1. **Tailored Silhouettes**: A shift towards slimmer fits with sharply tailored dark suits, as seen in collections from brands like Saint Laurent and Gucci.\n\n2. **Fetishwear Elements**: Designers are incorporating fetish-inspired details into their collections, blending innocence with edginess.\n\n3. **Indie Sleaze Revival**: The 2010s aesthetic is making a comeback, characterized by Y2K influences and a rebellious spirit, with designers like Nicola Brognano drawing inspiration from pop culture icons.\n\n4. **Wardrobe Dressing**: A focus on practical, mix-and-match pieces that can be integrated into everyday wardrobes, emphasizing comfort and versatility.\n\n5. **1980s Inspiration**: Bold colors, big shoulders, and playful patterns are returning, with designers like Anthony Vaccarello at Saint Laurent

In [ ]:
print(result['output'])

The latest fashion trends for 2026 showcase a mix of styles and influences. Key trends include:

1. **Tailored Silhouettes**: A shift towards slimmer fits with sharply tailored dark suits, as seen in collections from brands like Saint Laurent and Gucci.

2. **Fetishwear Elements**: Designers are incorporating fetish-inspired details into their collections, blending innocence with edginess.

3. **Indie Sleaze Revival**: The 2010s aesthetic is making a comeback, characterized by Y2K influences and a rebellious spirit, with designers like Nicola Brognano drawing inspiration from pop culture icons.

4. **Wardrobe Dressing**: A focus on practical, mix-and-match pieces that can be integrated into everyday wardrobes, emphasizing comfort and versatility.

5. **1980s Inspiration**: Bold colors, big shoulders, and playful patterns are returning, with designers like Anthony Vaccarello at Saint Laurent leading the charge.

6. **Bourgeois Aesthetic**: A modern take on classic Parisian style, featur

In [ ]:
# Reset conversation memory in case you need
memory.clear()
print("Conversation history cleared.")

Conversation history cleared.


In [ ]:
# Invoke the agent with chat history
# Let's try a query based on amazon data & check if 'amazon_search' tool is triggered
result = agent_with_chat_history.invoke(
    {"input": "Does Amazon have these latest trends?"},
    config={"configurable": {"session_id": "<foo>"}},
)



> Entering new AgentExecutor chain...
```
Thought: Do I need to use a tool? Yes
Action: search_tavily
Action Input: latest trends on Amazon
```Web Result 1: 
 URL: https://www.darkroomagency.com/observatory/which-amazon-product-categories-are-growing-in-2026 Content: ### Baby Products

Baby products don’t behave like “trend categories.” They behave like trust categories. Shoppers want safety, clarity, and fewer surprises, so brands that communicate materials, standards, and real-world usage tend to win.

What keeps growing:

Cleaner ingredients and sensitive-skin positioning (where applicable)

Practical gear that solves a specific pain point

Consumables with repeat purchase behavior

Broader market research continues to show demand growth tied to premiumization and “health-first” product choices, which is consistent with what performs on Amazon in baby-related segments.

# How to Research Emerging Trends on Amazon [...] This article walks through the fastest-growing (and most consi

In [ ]:
# Invoke the agent with chat history
# Let's try a query based on recent data and check whether 'Tavily Search' is triggered
result = agent_with_chat_history.invoke(
    {"input": "what is 2 + 3?"},
    config={"configurable": {"session_id": "<foo>"}},
)



> Entering new AgentExecutor chain...
```
Thought: Do I need to use a tool? Yes
Action: add
Action Input: "2", "3"

ValidationError: 1 validation error for add
b
  Field required [type=missing, input_value={'a': '2", "3'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing

In [ ]:
print(result['output'])

It seems that the search for the latest fashion trends on Amazon yielded some results, but they may not directly reflect the specific trends for 2026. However, you can find various clothing items that align with current fashion trends, such as tailored silhouettes, oversized sweaters, and versatile pieces. For the most accurate and up-to-date offerings, I recommend checking Amazon directly for specific items that match the latest trends.


In [ ]:
# Invoke the agent with chat history
# Let's try a query based on previous history
result = agent_with_chat_history.invoke(
    {"input": "Suggest me some best fashion accessories from Amazon "},
    config={"configurable": {"session_id": "<foo>"}},
)



> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes  
Action: amazon_product_search  
Action Input: best fashion accessories  Result 1: Description <p>These Stylish Earring for Women and Girls are handcrafted with love and will add a touch of spirituality to your look. It is suitable with all outfits &amp; may be worn at any occasion.</p> <p>Redgem has huge collection of fine

Result 2: collection of fine jewellery of silver and precious gems for women.</p> <p>◆&nbsp;<strong>Pure Sterling Silver Earring for girls and Women&nbsp;</strong>makes it very simple and attractive.</p> <p>◆ <strong>Design</strong>: A Stylish Pure Sterling

Result 3: your style! Or with hundreds of other Skins designs, you can be sure to find one that you&rsquo;ll love, and that will show off your unique style!</p> <p><strong>Do You Want To Protect Your Device?</strong></p> <p>With our Skins your Device is

Result 4: from head to toe (barefoot)<br /><br />1. Originally Designed: All dre

In [ ]:
# Building a simple UI for the chatbot with agents
import gradio as gr

session_memory = {}

def get_memory(session_id: str):
    if session_id not in session_memory:
        session_memory[session_id] = InMemoryChatMessageHistory()
    return session_memory[session_id]

agent_with_chat_history = RunnableWithMessageHistory(
    agent_executor,
    get_memory,
    input_messages_key="input",
    history_messages_key="chat_history",
)

def chat_with_agent(user_input, session_id):
    response = agent_with_chat_history.invoke(
        {"input": user_input},
        config={"configurable": {"session_id": session_id}},
    )
    return response.get("output", str(response))

with gr.Blocks() as app:
    gr.Markdown("# 🤖 Review Genie - Agents & ReAct Framework")
    gr.Markdown("Ask a question and watch the ReAct agent choose tools. You can reuse the same session ID to keep memory.")

    session_id_box = gr.Textbox(label="Session ID", value="demo-session")
    input_box = gr.Textbox(label="Enter your query", placeholder="Ask something...")
    output_box = gr.Textbox(label="Response", lines=12)
    submit_button = gr.Button("Submit")

    submit_button.click(chat_with_agent, inputs=[input_box, session_id_box], outputs=output_box)

app.launch(debug=True, share=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://95f69a52213e74d8dd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)




> Entering new AgentExecutor chain...
```
Thought: Do I need to use a tool? Yes
Action: amazon_product_search
Action Input: good shoesResult 1: Title Mad Rock Remora Climbing Shoe - Men's Blue 9.5

Result 2: Description The Remora Climbing Shoe is Mad Rock's do-it-all slipper for climbers who can't have separate shoes for boulders, sport routes, and gyms. With a moderately stiff, slightly downturned design, the Remora performs on any climb at steep to

Result 3: Title adidas Men's Predator 18+ FG Firm Ground Soccer Cleats

Description adidas Predator 18+ FG- Black 7.5

Result 4: Title Kenneth Cole REACTION Men's Crespo Loafer B Shoe, Cognac, 10 M USDo I need to use a tool? No
Final Answer: Here are some good shoes available on Amazon:

1. **Mad Rock Remora Climbing Shoe - Men's**: A versatile climbing shoe suitable for various types of climbing, featuring a moderately stiff and slightly downturned design.

2. **adidas Men's Predator 18+ FG Firm Ground Soccer Cleats**: Designed for so

## Adding more features to the chatbot......

Now we extend the same ReAct idea with more tools. The key teaching point is that the **same reasoning loop** can decide among different capabilities:

- local Amazon retrieval,
- web search,
- and later, a weather API.


In [ ]:
# Create a multi-tool ReAct agent (same pattern, same prompt, same trace)
tools = [search_tavily, amazon_product_search]

multi_tool_react_agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
)

multi_tool_agent_executor = AgentExecutor(
    agent=multi_tool_react_agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
)

# Test the multi-tool system
query = "Compare the latest MacBook Air and Dell XPS laptops."
response = multi_tool_agent_executor.invoke({"input": query, "chat_history": []})
print(response["output"])




> Entering new AgentExecutor chain...
```
Thought: Do I need to use a tool? Yes
Action: search_tavily
Action Input: Compare the latest MacBook Air and Dell XPS laptopsWeb Result 1: 
 URL: https://www.techradar.com/computing/laptops/dell-xps-13-vs-macbook-air-which-is-the-king-of-laptops Content: In comparison, the MacBook Air also comes with a sleek chassis, which again looks breathtaking. It’s super light, weighing almost exactly the same as the XPS 13, making it a similarly great option to pop in your backpack and take on the go. The MacBook Air is actually thinner than the XPS 13, measuring in at 1.13cm to Dell’s 1.48cm - and while that sounds marginal, it's enough that you can notice the difference. [...] Although the MacBook Air technically won more categories, it’s hard to determine a clear overall winner between these two laptops. While the Dell XPS 13 decisively takes home the crown when it comes to battery life and display quality, the MacBook Air does offer very similar per

In [ ]:
# Another test query
query = "Which is more economical: MacBook Pro or MacBook Air?"
response = multi_tool_agent_executor.invoke({"input": query, "chat_history": []})
print(response["output"])




> Entering new AgentExecutor chain...
```
Thought: Do I need to use a tool? No
Final Answer: Generally, the MacBook Air is more economical than the MacBook Pro. The MacBook Air is designed to be a more budget-friendly option, offering a good balance of performance and price, while the MacBook Pro is aimed at users who need higher performance for tasks like video editing, graphic design, and software development, which typically comes at a higher cost. However, the best choice depends on your specific needs and budget. If you require more power and features, the MacBook Pro may be worth the investment, but for everyday tasks, the MacBook Air is usually the more economical choice.
```

> Finished chain.
Generally, the MacBook Air is more economical than the MacBook Pro. The MacBook Air is designed to be a more budget-friendly option, offering a good balance of performance and price, while the MacBook Pro is aimed at users who need higher performance for tasks like video editing, graphi

In [ ]:
# OpenWeatherMap API configuration
WEATHER_API_KEY = getpass("Enter your OpenWeatherMap API key: ")
BASE_URL = "https://api.openweathermap.org/data/2.5/weather"

Enter your OpenWeatherMap API key: ··········


In [ ]:
import requests

@tool
def get_weather(city: str) -> str:
    """Fetch real-time weather data for a city and suggest what to carry."""
    params = {
        "q": city,
        "appid": WEATHER_API_KEY,
        "units": "metric",
    }

    response = requests.get(BASE_URL, params=params, timeout=30)
    if response.status_code != 200:
        try:
            err = response.json().get("message", "Unknown error")
        except Exception:
            err = "Unknown error"
        return f"Error fetching weather data: {err}"

    data = response.json()
    temp = data["main"]["temp"]
    feels_like = data["main"].get("feels_like", temp)
    condition = data["weather"][0]["description"]

    if temp < 8:
        recommendation = "Carry a warm jacket, layers, and possibly gloves."
    elif temp < 18:
        recommendation = "Carry a light jacket or sweater."
    elif temp < 28:
        recommendation = "Light clothes should be fine, but bring an extra layer for the evening."
    else:
        recommendation = "Carry breathable summer clothes, sunglasses, and water."

    return (
        f"Current weather in {city}: {temp}°C"
        + (f", feels like {feels_like}°C" if feels_like is not None else "")
        + f", condition: {condition}. Recommendation: {recommendation}."
    )


In [ ]:
# Add weather as a third tool and test the extended ReAct agent
tools = [search_tavily, amazon_product_search, get_weather]

travel_react_agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
)

travel_agent_executor = AgentExecutor(
    agent=travel_react_agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
)



> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes  
Action: get_weather  
Action Input: Paris  Current weather in Paris: 16.77°C, feels like 15.73°C, condition: broken clouds. Recommendation: Carry a light jacket or sweater..Do I need to use a tool? Yes  
Action: amazon_product_search  
Action Input: travel essentials for Paris  Result 1: Title 100 clés des villes sœurs. Eu - Le Tréport - Mers-

Result 2: anywhere life takes you. 10 bottle each of Lavender, Rose, Tea Tree, Rosemary, Jasmine, Rajnigandha, Eucalyptus, Lemongrass, Orchid & Sandalwood fragrance.

Result 3: Description Transform your home, workplace or hotel room into your personal aromatherapy oasis! With elegantly designed diffusers and aroma diffuser oils, you can infuse essential oils into any setting and create a spa-like experience anywhere life

Result 4: Title Seven Tips to Survival: Overcoming the Speed Bumps on Your JourneyDo I need to use a tool? No  
Final Answer: For your trip to Par

In [ ]:
query = "I will travel to Paris now. What should I carry for the weather, and should I purchase anything from Amazon?"
response = travel_agent_executor.invoke({"input": query, "chat_history": []})
print(response["output"])